# Cloudflare credentials

Set these values in your local `.env` file or shell before running the notebook. Do not paste live tokens into the notebook.

Required variables: `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_S3_ENDPOINT`, `R2_BUCKET_NAME`, `CLOUDFLARE_ACCOUNT_ID`, `CLOUDFLARE_API_TOKEN`, `D1_DATABASE_ID`.


In [ ]:
import os
import polars as pl

# R2 Configuration
storage_options = {
    "aws_access_key_id": os.environ["R2_ACCESS_KEY_ID"],
    "aws_secret_access_key": os.environ["R2_SECRET_ACCESS_KEY"],
    "endpoint_url": os.environ["R2_S3_ENDPOINT"],
    "aws_region": os.getenv("AWS_REGION", "auto"),
}

bucket_name = os.environ["R2_BUCKET_NAME"]

# Scan the entire directory (Lazy Loading)
# Polars will only read the files/columns necessary when you call .collect()
df = (
    pl.scan_parquet(f"s3://{bucket_name}/futures/ohlcv/**/*.parquet", storage_options=storage_options)
    .collect()
)

ctx = pl.SQLContext(ohlcv_futures=df)


In [ ]:
df.shape

In [ ]:
query_bounds = """
SELECT
    expiry_date,
    MIN(datetime) AS first_transaction,
    MAX(datetime) AS last_transaction,
    COUNT(DISTINCT CAST(datetime AS DATE)) AS total_trading_days
FROM ohlcv_futures
WHERE stock_code = 'NIFTY'
GROUP BY expiry_date
"""
bounds_df = ctx.execute(query_bounds).collect()

In [ ]:
bounds_df